In [1]:
import numpy as np 
import pandas as pd 


In [2]:
df=pd.read_csv("Tweets.csv")


In [3]:
df.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [4]:
df.columns

Index(['textID', 'text', 'selected_text', 'sentiment'], dtype='object')

In [5]:
df.drop(columns=['textID','selected_text'],inplace=True)

In [6]:
df.isna().sum()

text         1
sentiment    0
dtype: int64

In [7]:
df.dropna(inplace=True)

In [8]:
df['sentiment'].value_counts()

sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64

In [9]:
df = df[df['sentiment'] != 'neutral']

In [10]:
import re
import string

In [11]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#\w+", "", text)      # remove hashtags
    text = re.sub(r"[0-9]", "", text)     # remove numbers
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation

    text = text.strip()

    return text

In [12]:
df['clean_tweet'] = df['text'].apply(clean_text)


In [13]:
df.head()

,text,sentiment,clean_tweet
1,Sooo SAD I will miss you here in San Diego!!!,negative,sooo sad i will miss you here in san diego
2,my boss is bullying me...,negative,my boss is bullying me
3,what interview! leave me alone,negative,what interview leave me alone
4,"Sons of ****, why couldn`t they put them on t...",negative,sons of why couldnt they put them on the rele...
6,2am feedings for the baby are fun when he is a...,positive,am feedings for the baby are fun when he is al...


In [14]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download('stopwords')
# nltk.download('wordnet')

In [15]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [16]:
df['processed_tweet'] = df['clean_tweet'].apply(preprocess)

In [17]:
df.head()

,text,sentiment,clean_tweet,processed_tweet
1,Sooo SAD I will miss you here in San Diego!!!,negative,sooo sad i will miss you here in san diego,sooo sad miss san diego
2,my boss is bullying me...,negative,my boss is bullying me,bos bullying
3,what interview! leave me alone,negative,what interview leave me alone,interview leave alone
4,"Sons of ****, why couldn`t they put them on t...",negative,sons of why couldnt they put them on the rele...,son couldnt put release already bought
6,2am feedings for the baby are fun when he is a...,positive,am feedings for the baby are fun when he is al...,feeding baby fun smile coo


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [19]:
X = df['processed_tweet']
y = df['sentiment']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

In [22]:
model = LogisticRegression()

model.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [23]:
y_pred = model.predict(X_test_vec)

In [24]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8649556981362664


In [25]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

In [26]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', LogisticRegression())
])

In [29]:
pipeline.fit(X_train,y_train)

,steps,"[('tfidf', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [30]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 86.59%


In [31]:
import joblib,os

In [33]:
def save_model(path="sentiment.pkl"):
    if os.path.exists(path):
        choice = input("Model already exists. Do you want to replace it? (y/n): ").strip().lower()
        if choice != "y":
            print("Model was not replaced.")
            return

    joblib.dump((pipeline), path)
    print("Model saved successfully.")
save_model()

Model saved successfully.
